# Two-dimensional triangular-mesh deposition

Ten beams are traced through a cylindrical hydro grid. Their sheet-resolved affine source is integrated exactly onto a separate circular triangle mesh by the canonical JAX overlap kernels.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.tri as mtri
import numpy as np
from matplotlib.colors import LogNorm

from pyGATH.fields import (
    build_circular_deposition_mesh_from_grid,
    deposit_simplicial_power_to_mesh,
    simplicialise_sheet_fields,
)
from pyGATH.io import load_simulation_config

root = Path.cwd().resolve()
if root.name == "examples":
    root = root.parent
simulation = load_simulation_config(
    root / "configs/example_configs/paper_s64_ten_beam_cylindrical_deposition.toml"
)

In [ ]:
with simulation.reporting():
    grid = simulation.build_grid()
    beams = simulation.load_beams()
    initial_rays = simulation.initialize_rays(grid, beams=beams)
    trace = simulation.trace_rays(initial_rays, grid)
    source = simplicialise_sheet_fields(
        trace.sheet_fields, dimension=2, fields="inverse_brems_deposition"
    )
    target = build_circular_deposition_mesh_from_grid(grid, maximum_angular_cells=80)
    resolved_deposition = deposit_simplicial_power_to_mesh(
        source, target, resolve_beam_sheets=True
    )
    deposition = resolved_deposition.total
print(f"{source.mesh.nsimplices:,} source triangles per sheet")
print(f"{target.ncells:,} target triangles")
print(f"conservation error={deposition.conservation_error:.3e} W")

In [ ]:
def positive_log_norm(arrays, dynamic_range=1.0e6):
    positive = np.concatenate(
        [np.asarray(array)[np.asarray(array) > 0.0] for array in arrays]
    )
    maximum = positive.max()
    minimum = max(positive.min(), maximum / dynamic_range)
    if minimum >= maximum:
        minimum = 0.5 * maximum
    return LogNorm(vmin=minimum, vmax=maximum)


def format_xy(axis):
    axis.set_aspect("equal")
    axis.set_xlabel(r"$x$ [$\mu$m]")
    axis.set_ylabel(r"$y$ [$\mu$m]")


source_triangulations = []
source_density = []
field_index = source.selection.inverse_brems_deposition
for sheet in range(2):
    positions = np.asarray(source.mesh.vertex_positions[0, sheet]) * 1.0e6
    valid = np.asarray(source.mesh.valid[0, sheet], dtype=bool)
    source_triangulations.append(
        mtri.Triangulation(
            positions[:, 0],
            positions[:, 1],
            triangles=np.asarray(source.mesh.connectivity),
            mask=~valid,
        )
    )
    source_density.append(np.asarray(source.vertex_values[0, sheet, :, field_index]))

target_positions = np.asarray(target.vertex_positions) * 1.0e6
target_triangulation = mtri.Triangulation(
    target_positions[:, 0],
    target_positions[:, 1],
    triangles=np.asarray(target.simplex_connectivity),
)
sheet_depositions = [
    resolved_deposition.select(beam_index=0, sheet_index=sheet) for sheet in range(2)
]

## Beam 1 source sheets

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)
for sheet, axis in enumerate(axes):
    axis.triplot(source_triangulations[sheet], color="0.25", linewidth=0.45)
    axis.set_title(f"Beam 1, sheet {sheet + 1}")
    format_xy(axis)
figure.suptitle("Source-sheet triangulations")
figure.tight_layout()

In [ ]:
source_norm = positive_log_norm(source_density)
figure, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)
for sheet, axis in enumerate(axes):
    image = axis.tripcolor(
        source_triangulations[sheet],
        source_density[sheet],
        shading="gouraud",
        norm=source_norm,
    )
    axis.triplot(source_triangulations[sheet], color="white", linewidth=0.2, alpha=0.35)
    axis.set_title(f"Beam 1, sheet {sheet + 1}")
    format_xy(axis)
figure.colorbar(image, ax=axes, label=r"volumetric deposition [W/m$^3$]", shrink=0.9)
figure.suptitle("Volumetric deposition on the source sheets")

## Deposition mesh and mapped power density

In [ ]:
figure, axis = plt.subplots(figsize=(7, 6))
axis.triplot(target_triangulation, color="0.25", linewidth=0.35)
format_xy(axis)
axis.set_title("Circular deposition mesh")
figure.tight_layout()

In [ ]:
sheet_power_density = [np.asarray(result.power_density) for result in sheet_depositions]
sheet_norm = positive_log_norm(sheet_power_density)
figure, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)
for sheet, axis in enumerate(axes):
    image = axis.tripcolor(
        target_triangulation,
        facecolors=sheet_power_density[sheet],
        shading="flat",
        norm=sheet_norm,
    )
    axis.set_title(f"Beam 1, sheet {sheet + 1}")
    format_xy(axis)
figure.colorbar(image, ax=axes, label=r"volumetric deposition [W/m$^3$]", shrink=0.9)
figure.suptitle("Sheet-resolved deposition on the target mesh")

In [ ]:
total_density = np.asarray(deposition.power_density)
figure, axis = plt.subplots(figsize=(7, 6))
image = axis.tripcolor(
    target_triangulation,
    facecolors=total_density,
    shading="flat",
    norm=positive_log_norm([total_density]),
)
format_xy(axis)
axis.set_title("All-beam deposition on the target mesh")
figure.colorbar(image, ax=axis, label=r"volumetric deposition [W/m$^3$]")
figure.tight_layout()